<a href="https://colab.research.google.com/github/chisangachileshe623-ship-it/Africa-Digital-Divide/blob/main/01_africa_digital_divide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print ("Africa's Digital Divide")
print ("Project 1")

Africa's Digital Divide
Project 1


In [ ]:
import requests

url = "https://api.worldbank.org/v2/country/all/indicator/IT.NET.USER.ZS?format=json&per_page=20000"

response = requests.get(url)

print("Status code:", response.status_code)
print("Response type:", response.headers.get("content-type"))
print(response.text[:500])

Status code: 200
Response type: application/json;charset=utf-8
[{"page":1,"pages":1,"per_page":20000,"total":17490,"sourceid":"2","lastupdated":"2026-07-13"},[{"indicator":{"id":"IT.NET.USER.ZS","value":"Individuals using the Internet (% of population)"},"country":{"id":"ZH","value":"Africa Eastern and Southern"},"countryiso3code":"AFE","date":"2025","value":30.4,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"IT.NET.USER.ZS","value":"Individuals using the Internet (% of population)"},"country":{"id":"ZH","value":"Africa Eastern and Southern"},"c


In [ ]:
import requests
import pandas as pd

indicators = {
    "IT.NET.USER.ZS": "internet_users",
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "EG.ELC.ACCS.ZS": "electricity_access",
    "SP.URB.TOTL.IN.ZS": "urban_population"
}

dataframes = []

for indicator, name in indicators.items():

    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}?format=json&per_page=20000"

    response = requests.get(url)

    response.raise_for_status()

    result = response.json()

    observations = result[1]

    df = pd.DataFrame(observations)

    df['country'] = df['country'].apply(lambda x: x['value'] if isinstance(x, dict) else x)

    df = df[[
        "countryiso3code",
        "country",
        "date",
        "value"
    ]]

    df = df.rename(columns={
        "value": name
    })

    dataframes.append(df)


data = dataframes[0]

for df in dataframes[1:]:
    data = data.merge(
        df,
        on=["countryiso3code", "country", "date"],
        how="outer"
    )

data = data.rename(columns={"date": "year"})

data["year"] = pd.to_numeric(data["year"])

data = data[
    (data["year"] >= 2000) &
    (data["year"] <= 2024)
]

print("Dataset shape:", data.shape)

data.head()

Dataset shape: (6625, 7)


,countryiso3code,country,year,internet_users,gdp_per_capita,electricity_access,urban_population
40,,High income,2000,NaN,30253.840917,99.441643,76.124359
41,,High income,2001,NaN,30574.410855,99.466443,76.383580
42,,High income,2002,NaN,30920.087105,99.486329,76.705729
43,,High income,2003,NaN,31459.581730,99.528425,77.060638
44,,High income,2004,NaN,32386.375080,99.523189,77.409597


In [ ]:
data.shape

(6625, 7)

In [ ]:
data.columns

Index(['countryiso3code', 'country', 'year', 'internet_users',
       'gdp_per_capita', 'electricity_access', 'urban_population'],
      dtype='object')

In [ ]:
data["country"].nunique()

265